In [ ]:
import os
import anndata as ad
import numpy as np
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import omicverse as ov
import scvi
from scvi.model.utils import mde

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

In [2]:
sc.settings.set_figure_params(dpi=100, frameon=False)
sc.set_figure_params(dpi=100)
sc.set_figure_params(figsize=(3, 3))
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (3, 3)

In [ ]:
# Change the working directory to the Garfield folder (if needed)
os.chdir('/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250729_scpoli_optimization_v3')
os.getcwd()

In [4]:
# Process query datasets from folder
query_folder = '/storage2/liuxiaodongLab/fanxueying/mayanalysis/2024Aug/garfield/in_vitro_embryo_models/processed/data'
# List all h5ad files in the directory
h5ad_files = [os.path.join(query_folder, f) for f in os.listdir(query_folder) if f.startswith('corrected_processed_')]

In [ ]:
# Read each h5ad file into an AnnData object
adata_list = []
for file in h5ad_files:
    print(f"Reading file: {file}")
    adata = sc.read_h5ad(file)  # Read the h5ad file into AnnData
    adata_list.append(adata)  # Add AnnData object to the list

In [ ]:
# add scpoli lineage results
# Define the output directory
output_dir = '/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250729_scpoli_optimization_v3/lineage_batch_enhanced_pred_2ndround/figures'

# Step 1: List all CSV files in the directory
scpoli_files = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.endswith('.csv')]

# Step 2: Initialize an empty dictionary to store the results
scpoli_tables = {}

# Step 3: Process each file
for file in scpoli_files:
    # Read the CSV file into a DataFrame
    scpoli = pd.read_csv(file)
    
    # Extract the desired substring from the file name
    file_name = os.path.basename(file)  # Get the file name without the path
    new_name = file_name.replace('corrected_processed_', '')  # Remove "corrected_processed_"
    new_name = new_name.replace('.h5ad_scPoli_query.csv', '')  # Remove ".h5ad"
    
    # Store the DataFrame in the dictionary with the new name
    scpoli_tables[new_name] = scpoli

# Now `scpoli_tables` is a dictionary where keys are the processed file names and values are the DataFrames

# List of attributes to be extracted from Garfield results
attri = ["lineage_pred", "lineage_uncert"]

# Process each AnnData object and merge with Garfield results
for i, dataset in enumerate(adata_list):
    # Extract the dataset name from the file name
    name = os.path.basename(h5ad_files[i]).replace('.h5ad', '')  
    name = name.replace('corrected_processed_', '')  
    
    # Get the corresponding scpoli results
    garf = scpoli_tables.get(name)
    
    if garf is not None:
        # Check if 'X' is the first column or index
        if garf.columns[0] != 'X':  
            print(f"Warning: The first column is not 'X' for {name}. Renaming the first column.")
            garf.rename(columns={garf.columns[0]: 'X'}, inplace=True)
        
        # Set the first column as index
        garf.set_index('X', inplace=True)
        garf.index = garf.index.str.replace('-1-1$', '-1', regex=True)  # Fix extra "-1"

        # Check if indices match
        if not garf.index.equals(dataset.obs_names):
            print(f"Warning: Indices do not match for {name}. Attempting to fix...")
            
            # Standardize index formatting
            garf.index = garf.index.astype(str)  # Ensure string format
            dataset.obs_names = dataset.obs_names.astype(str)
            
            # Try stripping potential "-1" suffix
            garf.index = garf.index.str.replace('-1$', '', regex=True)
            dataset.obs_names = dataset.obs_names.str.replace('-1$', '', regex=True)
            
            # Check again after fixing
            if not garf.index.equals(dataset.obs_names):
                print(f"Error: Indices still do not match for {name}. Skipping this dataset.")
                continue

        # Extract required columns
        garf = garf[attri]
        
        # Remove filtered cells
        dataset = dataset[dataset.obs_names.isin(garf.index), :]
        
        # Ensure row names match
        garf = garf.loc[dataset.obs_names, :]
        
        # Add prefix to column names
        garf.columns = [f"human_ref_{col}" for col in garf.columns]

        # Merge into AnnData metadata
        dataset.obs = pd.concat([dataset.obs, garf], axis=1)

        # Update dataset in the list
        adata_list[i] = dataset
    else:
        print(f"No scpoli data found for {name}")


In [ ]:
# add scpoli reanno results
# Define the output directory
output_dir = '/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250729_scpoli_optimization_v3/reanno_batch_enhanced_pred_2ndround/figures'

# Step 1: List all CSV files in the directory
scpoli_files = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.endswith('.csv')]

# Step 2: Initialize an empty dictionary to store the results
scpoli_tables = {}

# Step 3: Process each file
for file in scpoli_files:
    # Read the CSV file into a DataFrame
    scpoli = pd.read_csv(file)
    
    # Extract the desired substring from the file name
    file_name = os.path.basename(file)  # Get the file name without the path
    new_name = file_name.replace('corrected_processed_', '')  # Remove "corrected_processed_"
    new_name = new_name.replace('.h5ad_scPoli_query.csv', '')  # Remove ".h5ad"
    
    # Store the DataFrame in the dictionary with the new name
    scpoli_tables[new_name] = scpoli

# Now `scpoli_tables` is a dictionary where keys are the processed file names and values are the DataFrames

# List of attributes to be extracted from Garfield results
attri = ["reanno_pred", "reanno_uncert"]

# Process each AnnData object and merge with Garfield results
for i, dataset in enumerate(adata_list):
    # Extract the dataset name from the file name
    name = os.path.basename(h5ad_files[i]).replace('.h5ad', '')  
    name = name.replace('corrected_processed_', '')  
    
    # Get the corresponding scpoli results
    garf = scpoli_tables.get(name)
    
    if garf is not None:
        # Check if 'X' is the first column or index
        if garf.columns[0] != 'X':  
            print(f"Warning: The first column is not 'X' for {name}. Renaming the first column.")
            garf.rename(columns={garf.columns[0]: 'X'}, inplace=True)
        
        # Set the first column as index
        garf.set_index('X', inplace=True)
        garf.index = garf.index.str.replace('-1-1$', '-1', regex=True)  # Fix extra "-1"

        # Check if indices match
        if not garf.index.equals(dataset.obs_names):
            print(f"Warning: Indices do not match for {name}. Attempting to fix...")
            
            # Standardize index formatting
            garf.index = garf.index.astype(str)  # Ensure string format
            dataset.obs_names = dataset.obs_names.astype(str)
            
            # Try stripping potential "-1" suffix
            garf.index = garf.index.str.replace('-1$', '', regex=True)
            dataset.obs_names = dataset.obs_names.str.replace('-1$', '', regex=True)
            
            # Check again after fixing
            if not garf.index.equals(dataset.obs_names):
                print(f"Error: Indices still do not match for {name}. Skipping this dataset.")
                continue

        # Extract required columns
        garf = garf[attri]
        
        # Remove filtered cells
        dataset = dataset[dataset.obs_names.isin(garf.index), :]
        
        # Ensure row names match
        garf = garf.loc[dataset.obs_names, :]
        
        # Add prefix to column names
        garf.columns = [f"human_ref_{col}" for col in garf.columns]

        # Merge into AnnData metadata
        dataset.obs = pd.concat([dataset.obs, garf], axis=1)

        # Update dataset in the list
        adata_list[i] = dataset
    else:
        print(f"No scpoli data found for {name}")


In [ ]:
# Merge all AnnData objects into one
# We use `anndata.concat()` to merge them, making sure the 'obs' and 'var' fields align
combined_adata = ad.concat(adata_list, label='batch', join='outer')

# Print the shape of the merged AnnData to verify
print(f"Combined AnnData shape: {combined_adata.shape}")


In [ ]:
combined_adata

In [ ]:
set(combined_adata.obs["orig.ident"])

In [ ]:
print(combined_adata.X)

In [12]:
combined_adata.layers["counts"] = combined_adata.X.copy()

In [13]:
sc.settings.seed = 42
sc.pp.normalize_total(combined_adata, target_sum=1e4)
sc.pp.log1p(combined_adata)
combined_adata.layers["logcounts"] = combined_adata.X.copy()
sc.pp.highly_variable_genes(combined_adata, n_top_genes=2000, flavor="cell_ranger", batch_key="orig.ident")
sc.tl.pca(combined_adata, n_comps=30, use_highly_variable=True)

In [14]:
adata_hvg = combined_adata[:, combined_adata.var.highly_variable].copy()

In [ ]:
adata_hvg

In [ ]:
####Unintegrated
combined_adata.obsm["Unintegrated"] = adata_hvg.obsm["X_pca"]
adata_hvg.obsm["Unintegrated"] = adata_hvg.obsm["X_pca"]
sc.pp.neighbors(combined_adata, use_rep="Unintegrated",random_state=42)
sc.tl.leiden(combined_adata, resolution=0.5,key_added=f"Unintegrated_res_0.5",random_state=42)
sc.tl.umap(combined_adata,random_state=42)
combined_adata.obsm['X_Unintegrated'] = combined_adata.obsm['X_umap']
sc.pl.umap(combined_adata, color=['orig.ident'], save="Unintegrated_orig_ident.pdf")
sc.pl.umap(combined_adata, color=['stage'], save="Unintegrated_stage.pdf")
sc.pl.umap(combined_adata, color=['human_ref_lineage_pred'], save="human_ref_lineage_pred.pdf")

In [ ]:
###scVI
scvi.model.SCVI.setup_anndata(adata_hvg, layer="counts", batch_key="orig.ident")
vae = scvi.model.SCVI(adata_hvg, gene_likelihood="nb", n_layers=2, n_latent=30)
vae.train()

In [18]:
combined_adata.obsm["scVI"] = vae.get_latent_representation()

In [ ]:
###scANVI
lvae = scvi.model.SCANVI.from_scvi_model(
    vae,
    adata=adata_hvg,
    labels_key="human_ref_lineage_pred",
    unlabeled_category="Unknown",
)
lvae.train(max_epochs=20, n_samples_per_label=100)

In [20]:
combined_adata.obsm["scANVI"] = lvae.get_latent_representation()

In [ ]:
adata_hvg.obsm["scANVI"] = combined_adata.obsm["scANVI"]
sc.pp.neighbors(combined_adata, use_rep="scANVI",random_state=42)
sc.tl.leiden(combined_adata, resolution=0.5,key_added=f"scANVI_res_0.5",random_state=42)
sc.tl.umap(combined_adata,random_state=42)
combined_adata.obsm['X_scANVI'] = combined_adata.obsm['X_umap']
adata_hvg.obsm['X_scANVI'] = combined_adata.obsm['X_scANVI']
sc.pl.umap(combined_adata, color=['orig.ident'],save="scANVI_orig.ident.pdf")
sc.pl.umap(combined_adata, color=['stage'],save="scANVI_stage.pdf")

In [ ]:
sc.pl.umap(combined_adata, color=['human_ref_lineage_pred'],save="scpoli_transferred_lineage_3round.pdf")

In [23]:
combined_adata.raw.var.rename(columns={'_index': 'index'}, inplace=True)
combined_adata.write_h5ad(filename="embryo_model_integration_scPoli_3round.h5ad")

In [ ]:
# Define the list of genes you want to plot
genes_to_plot = ["POU5F1","NANOG",#epiblast
                 "SOX2", "TTYH1", #Neural_ectoderm
                 "GATA3","TFAP2A",
                 "TBXT",  "CDX1","PDGFRA","APOA2","FOXA2","NANOS3",
                "PECAM1","HBZ","PTPRC",
                "GABRP","HEY1","COL6A1","COL6A2",]  # Replace "GENE2" and "GENE3" with the actual gene names

# Plot the UMAP embedding for the specified genes
sc.pl.embedding(
    combined_adata,
    basis='X_umap',
    color=genes_to_plot,
    use_raw=False,
    cmap=sns.cubehelix_palette(dark=0, light=.9, as_cmap=True)
)

In [25]:
adata = sc.read_h5ad('embryo_model_integration_scPoli_3round.h5ad') 

In [ ]:
adata

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# 1. Define the color palette (corrected Python syntax)
# Note: Dictionaries in Python use {} and key-value pairs are separated by colons
lineage_color_mapping = {
    "Amniotic_ecto": "#1f77b4",    # Blue
    "Notochord": "#aa40fc",        # Purple
    "Endoderm": "#ff7f0e",         # Orange
    "PGC": "#8c564b",              # Brown
    "ExE_endo": "#279e68",         # Green
    "Primitive.streak": "#e377c2", # Pink
    "NMP": "#d62728",              # Red
    "TE_TrB": "#b5bd61",           # Yellow-Green
    "epi": "#17becf",              # Cyan
    "hemogenic": "#aec7e8",        # Light Blue
    "meso_Exe.meso": "#ffbb78",    # Orange-Yellow
    "neural_ecto": "#98df8a"       # Light Green
}

# 2. Assume 'adata' is your AnnData object and it has a UMAP computed
#    and a column named 'lineage_pred' (or whatever your lineage prediction column is named)
#    in adata.obs

# Example column name - replace with the actual name in your adata object
lineage_column_name = 'human_ref_lineage_pred' # <--- CHANGE THIS TO YOUR ACTUAL COLUMN NAME

# Check if the column exists
if lineage_column_name not in adata.obs.columns:
    raise ValueError(f"Column '{lineage_column_name}' not found in adata.obs")

# 3. Filter the color mapping to only include keys present in the data
#    This prevents errors if some categories are missing in the data
# Get unique categories from the data (excluding NaN)
data_categories = set(adata.obs[lineage_column_name].dropna().unique())
# Filter the color mapping
available_colors = {k: v for k, v in lineage_color_mapping.items() if k in data_categories}

# Optional: Print which colors are being used
print("Available colors for plotting:")
for cat, color in available_colors.items():
    print(f"  {cat}: {color}")

# 4. Ensure the lineage column is categorical with categories ordered
#    (Optional but good practice for consistent plotting)
# Define the order based on the keys in your color mapping (or a specific order you want)
# Make sure to only include categories that are present in the data
ordered_categories = [k for k in lineage_color_mapping.keys() if k in data_categories]

# Update the categorical data in adata
adata.obs[lineage_column_name] = adata.obs[lineage_column_name].astype("category")
adata.obs[lineage_column_name] = adata.obs[lineage_column_name].cat.reorder_categories(ordered_categories, ordered=True)

# 5. Plot the UMAP using the defined color palette
# Create the plot
plt.figure(figsize=(8, 6)) # Adjust figure size as needed

# Use sc.pl.umap with the palette argument
sc.pl.umap(
    adata,
    color=['human_ref_lineage_pred'],  # Column to color by
    palette=available_colors,   # Dictionary mapping categories to colors
    title="UMAP colored by Lineage Prediction",
    show=True,                  # Set to False if you want to save instead of display
    frameon=False,               # Remove the frame around the plot (optional)
    save="scpoli_transferred_lineage_3round.pdf"
)

# If you want to save the plot instead of (or in addition to) showing it:
# plt.savefig("umap_lineage_plot.png", dpi=300, bbox_inches='tight')
# plt.close() # Close the figure to free memory